# Exercício Prático — Aula 08: Web Scraping com BeautifulSoup

**Análise e Ciência de Dados** (Especialização, IFRN) — Unidade 2, Aula 08

**Entrega:** este notebook **executado** (com as saídas visíveis) + os arquivos CSV/JSON gerados, compactados em um `.zip`, na tarefa *Exercício Prático — Aula 08* do Moodle. **Prazo: 24/10/2026 (23h59).**

Preencha os trechos marcados com `# TODO`. Use o notebook `Aula08_Web_Scraping_BeautifulSoup.ipynb` como referência — os padrões de código são os mesmos.
Coloque este notebook na mesma pasta da subpasta `site_resenhas/` (vem no pacote da aula).

In [1]:
import json
import subprocess
import sys
import time
from io import StringIO
from pathlib import Path
from urllib.parse import urljoin

import pandas as pd
import requests
from bs4 import BeautifulSoup

SAIDA = Path("dados_exercicio")
SAIDA.mkdir(exist_ok=True)
CABECALHOS = {"User-Agent": "IURI-PEREIRA-AulaACD/1.0 (exercicio educacional)"}  # TODO: coloque seu nome

def baixar(url):
    resp = requests.get(url, headers=CABECALHOS, timeout=10)
    resp.raise_for_status()
    resp.encoding = "utf-8"
    return BeautifulSoup(resp.text, "html.parser")

## Parte A — Estante Potiguar (site local)

1. Coloque o site no ar (célula abaixo) e raspe **todas as resenhas**, seguindo o link "Próxima" (`li.proxima a`) e esperando 1 s entre as páginas.
2. Monte um DataFrame com: `livro`, `autor`, `nota`, `leitor`, `data`, `comentario`, `util` (use `None` quando o contador não existir).

### Coloquei a parte de extração de resenhas em uma função para facilitar a paginação

In [2]:
def extrair_resenhas(el):
    registros = []
    for article in el.select("article.resenha"):
        util = article.select_one(".util")
        registros.append({
            "id": int(article["data-id"]),
            "livro": article.select_one("h2.livro").get_text(strip=True),
            "autor": article.select_one("p.autor").get_text(strip=True),
            "nota": int(article.select_one("div.nota")["data-nota"]),
            "leitor": article.select_one("p.leitor .nome").get_text(strip=True),
            "data": article.select_one("time")["datetime"],
            "comentario": article.select_one("p.comentario").get_text(strip=True),
            "util": int(util.get_text().split()[0]) if util else None,
        })
    return registros

In [3]:
servidor = subprocess.Popen([sys.executable, "-m", "http.server", "8000", "--directory", "site_resenhas"],
                            stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(1.5)
BASE = "http://localhost:8000/"

resenhas = []
url = BASE
resenhas = extrair_resenhas(baixar(url))

df_res = pd.DataFrame(resenhas)
print(len(df_res), "resenhas")
print(json.dumps(resenhas[0], indent=2, ensure_ascii=False))

8 resenhas
{
  "id": 1,
  "livro": "Dom Casmurro",
  "autor": "Machado de Assis",
  "nota": 5,
  "leitor": "Ana P.",
  "data": "2026-09-02",
  "comentario": "Releio a cada cinco anos e sempre mudo de opinião sobre Capitu. Narrador mais suspeito da literatura brasileira.",
  "util": 31
}


### Paginação

In [29]:
def raspar_paginado(url, pausa=1):
    """Segue li.proxima a até acabar; retorna todas as resenhas."""
    registros = []
    while url:
        sopa = baixar(url)
        registros.append(extrair_resenhas(sopa))
        proxima = sopa.select_one("li.proxima a")
        url = urljoin(url, proxima["href"]) if proxima else None # pyright: ignore[reportArgumentType]
        if url:
            time.sleep(pausa)
    return registros

resenhas = raspar_paginado(BASE)
df_res = pd.DataFrame(resenhas)
print(len(df_res), "resenhas em todas as páginas")
print(json.dumps(resenhas, indent=2, ensure_ascii=False))

3 resenhas em todas as páginas
[
  [
    {
      "id": 1,
      "livro": "Dom Casmurro",
      "autor": "Machado de Assis",
      "nota": 5,
      "leitor": "Ana P.",
      "data": "2026-09-02",
      "comentario": "Releio a cada cinco anos e sempre mudo de opinião sobre Capitu. Narrador mais suspeito da literatura brasileira.",
      "util": 31
    },
    {
      "id": 2,
      "livro": "O Cortiço",
      "autor": "Aluísio Azevedo",
      "nota": 4,
      "leitor": "Bruno M.",
      "data": "2026-09-03",
      "comentario": "Retrato forte e coletivo da vida urbana no século XIX. Algumas descrições são longas, mas valem a pena.",
      "util": 12
    },
    {
      "id": 3,
      "livro": "Iracema",
      "autor": "José de Alencar",
      "nota": 3,
      "leitor": "Carla R.",
      "data": "2026-09-04",
      "comentario": "Linguagem belíssima, quase um poema em prosa. Achei a história romantizada demais para o meu gosto.",
      "util": 7
    },
    {
      "id": 4,
      "livro": "O

3. Calcule, **por autor**, o número de resenhas e a nota média (ordenado da maior para a menor média).
4. Mostre a resenha **mais útil** (maior valor de `util`): livro, leitor e quantidade.
5. Leia a tabela `table#ranking` de `ranking.html` com `pd.read_html(..., decimal=",")` e confira se as médias por livro batem com as que você calcularia a partir das resenhas.

In [9]:
# TODO: agrupamento por autor
por_autor = (
    df_res.groupby("autor")
          .agg(resenhas=("nota", "size"), nota_media=("nota", "mean"))
          .sort_values("nota_media", ascending=False)
)
display(por_autor.style.format(subset=["nota_media"], precision=2, decimal=",", thousands="."))

# TODO: resenha mais útil
mais_util = df_res.loc[df_res["util"].idxmax()] # type: ignore
print(f"Mais útil: '{mais_util['livro']}' por {mais_util['leitor']} — {int(mais_util['util'])} acharam útil")


# TODO: tabela de ranking com pd.read_html e comparação
html_ranking = str(baixar(urljoin(BASE, "ranking.html")))  # baixar() já força UTF-8
ranking = pd.read_html(StringIO(html_ranking), attrs={"id": "ranking"}, decimal=",", thousands=".")[0]

,resenhas,nota_media
autor,,
Machado de Assis,4,"4,75"
Aluísio Azevedo,1,"4,00"
Lima Barreto,1,"4,00"
José de Alencar,2,"2,50"


Mais útil: 'Dom Casmurro' por Ana P. — 31 acharam útil


## Parte B — Books to Scrape (site real de treino)

6. Escolha **uma categoria com pelo menos 2 páginas** (lista no enunciado da tarefa; ex.: *Mystery*, *Romance*, *Fantasy*, *Food and Drink*). Os links das categorias estão em `div.side_categories ul li ul li a` na página inicial.
7. Raspe **todos os livros da categoria**, seguindo `li.next a` até o fim e esperando 1 s entre as páginas. Para cada livro guarde: `titulo` (atributo `title`), `preco` (float, sem o £), `nota` (1 a 5, a partir da classe `star-rating`) e `link` **absoluto** (use `urljoin` — nas categorias o `href` começa com `../../../`).

In [5]:
CATEGORIA = "Mystery"   # TODO: escolha a sua
NOTAS = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}

inicio = baixar("https://books.toscrape.com/index.html")
# TODO: encontrar o link da CATEGORIA na barra lateral

livros = []
# TODO: laço de paginação na categoria (li.next a), com time.sleep(1)

df_livros = pd.DataFrame(livros)
print(len(df_livros), "livros na categoria", CATEGORIA)

0 livros na categoria Mystery


8. Calcule: total de livros, preço médio, distribuição das notas e os **3 livros mais bem avaliados** (desempate pelo **menor** preço).
9. Abra a página de detalhe de **um** desses livros e extraia a tabela `table.table-striped` em um dicionário.

In [6]:
# TODO: estatísticas da categoria

# TODO: tabela da página de detalhe de um livro


## Parte C — Salvar e refletir

10. Salve em `dados_exercicio/`: as resenhas (CSV) e os livros da categoria (CSV + JSON com **proveniência**: fonte, URL, data da coleta).
11. Responda na célula Markdown abaixo (3 a 5 linhas): que cuidados éticos e legais você tomou (robots.txt, identificação, intervalo entre requisições, dados pessoais/LGPD)? O que mudaria se o site **não** permitisse a coleta?

In [7]:
# TODO: salvar CSV/JSON

servidor.terminate()

**Reflexão ética (escreva aqui):**

...